# Stage 2.5 - Inference

Load trained models from Stage 2 and run **alpha predictions on any date range**. Used for:
- Production-style inference ("what should we hold tomorrow?")
- Walk-forward backtest predictions
- Q-Atlas demo (predict on a specific historical date)

**Prereq:** `02_alpha_models.ipynb` has been run — `workspace/models/*` exist.

In [1]:
import os, sys, json, time, pickle
import numpy as np
import pandas as pd
import requests
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

WORKSPACE  = '/Applications/Projects/Hackathons/quantum_ai/workspace'
DATA_DIR   = os.path.join(WORKSPACE, 'data')
MODELS_DIR = os.path.join(WORKSPACE, 'models')

# ====== Choose what to predict on ======
INFERENCE_MODE  = 'last_n_days'   # 'last_n_days' | 'date_range' | 'specific_date'
N_DAYS          = 30              # used if mode='last_n_days'
DATE_FROM       = '2019-06-01'    # used if mode='date_range'
DATE_TO         = '2019-12-31'
SPECIFIC_DATE   = '2020-03-12'    # used if mode='specific_date' (COVID crash example)

# Which models to run
RUN_LIGHTGBM = True
RUN_VQC      = True
RUN_XPYQ     = True
RUN_CHRONOS  = False              # slow, off by default for inference

print(f'Models dir: {MODELS_DIR}')
print(f'Mode: {INFERENCE_MODE}')

Models dir: /Applications/Projects/Hackathons/quantum_ai/workspace/models
Mode: last_n_days


## 1. Load schema + master dataset

In [2]:
# Load schema (same features Stage 2 trained on)
with open(os.path.join(MODELS_DIR, 'signal_cols.json')) as f:
    schema = json.load(f)
SIGNAL_COLS = schema['signal_cols']
TARGET      = schema['target']
TICKERS     = schema['tickers']
print(f'{len(SIGNAL_COLS)} signal cols, target={TARGET}, {len(TICKERS)} tickers')

# Load master dataset
master = pd.read_parquet(os.path.join(DATA_DIR, 'master_dataset.parquet'))
master['timestamp'] = pd.to_datetime(master['timestamp'])
master = master.set_index(['timestamp', 'ticker'])
print(f'Master dataset loaded: {master.shape}')

# Select inference window
ts = master.index.get_level_values('timestamp')
all_dates = sorted(ts.unique())

if INFERENCE_MODE == 'last_n_days':
    target_dates = all_dates[-N_DAYS:]
elif INFERENCE_MODE == 'date_range':
    target_dates = [d for d in all_dates if pd.Timestamp(DATE_FROM) <= d <= pd.Timestamp(DATE_TO)]
elif INFERENCE_MODE == 'specific_date':
    target_dates = [pd.Timestamp(SPECIFIC_DATE)]
else:
    raise ValueError(f'Unknown mode {INFERENCE_MODE}')

inference_mask = ts.isin(target_dates)
X_infer = master.loc[inference_mask, SIGNAL_COLS]
print(f'Inference window: {len(target_dates)} dates -> {len(X_infer):,} rows ({len(TICKERS)} tickers each)')

28 signal cols, target=fwd_return_5d, 20 tickers
Master dataset loaded: (37460, 45)
Inference window: 30 dates -> 600 rows (20 tickers each)


## 2. LightGBM inference

In [3]:
lgb_pred = pd.Series(np.nan, index=X_infer.index, name='lgb_pred')

if RUN_LIGHTGBM:
    import lightgbm as lgb
    booster = lgb.Booster(model_file=os.path.join(MODELS_DIR, 'lgb_alpha.txt'))
    lgb_pred[:] = booster.predict(X_infer)
    print(f'LightGBM: predicted {len(lgb_pred):,} rows')
    display(lgb_pred.head(10).to_frame())
else:
    print('LightGBM disabled')

LightGBM: predicted 600 rows


lgb_pred
timestamp  ticker          
2020-04-29 AAPL    0.000926
           MSFT   -0.007057
           GOOGL   0.009064
           NVDA    0.064054
           JPM     0.016931
           BAC     0.015187
           V       0.030086
           MA      0.015338
           JNJ     0.013045
           PFE     0.002397

## 3. VQC inference

In [4]:
vqc_pred = pd.Series(np.nan, index=X_infer.index, name='vqc_pred')

if RUN_VQC:
    import torch
    import torch.nn as nn
    import pennylane as qml

    # Load VQC metadata + scaler + weights
    with open(os.path.join(MODELS_DIR, 'vqc_features.json')) as f:
        vqc_meta = json.load(f)
    vqc_features = vqc_meta['features']
    N_QUBITS = vqc_meta['n_qubits']
    N_LAYERS = vqc_meta['n_layers']
    with open(os.path.join(MODELS_DIR, 'vqc_scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)

    # Rebuild same circuit
    dev = qml.device('default.qubit', wires=N_QUBITS)
    @qml.qnode(dev, interface='torch')
    def vqc_circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
        qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
        return qml.expval(qml.PauliZ(0))

    class VQCRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.qlayer = qml.qnn.TorchLayer(vqc_circuit, {'weights': (N_LAYERS, N_QUBITS)})
        def forward(self, x):
            return torch.stack([self.qlayer(xi) for xi in x]).reshape(-1, 1)

    vqc = VQCRegressor()
    vqc.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'vqc_alpha.pt')))
    vqc.eval()

    # Prepare inputs
    X_vqc = X_infer[vqc_features].fillna(X_infer[vqc_features].median())
    X_vqc_norm = scaler.transform(X_vqc) * (np.pi / 3)
    X_vqc_norm = np.clip(X_vqc_norm, -np.pi, np.pi)

    print(f'VQC inference on {len(X_vqc_norm):,} samples (4 qubits, may take ~30s for ~600 rows)...')
    t0 = time.time()
    with torch.no_grad():
        X_torch = torch.tensor(X_vqc_norm, dtype=torch.float32)
        batch_size = 200
        preds = []
        for i in range(0, len(X_torch), batch_size):
            preds.append(vqc(X_torch[i:i+batch_size]).numpy())
        vqc_pred_arr = np.concatenate(preds).flatten()
    vqc_pred[:] = vqc_pred_arr
    print(f'VQC done in {time.time() - t0:.1f}s')
    display(vqc_pred.head(10).to_frame())
else:
    print('VQC disabled')

VQC inference on 600 samples (4 qubits, may take ~30s for ~600 rows)...
VQC done in 1.2s


vqc_pred
timestamp  ticker          
2020-04-29 AAPL   -0.001095
           MSFT   -0.000908
           GOOGL  -0.001061
           NVDA    0.001706
           JPM     0.000382
           BAC     0.000983
           V      -0.001460
           MA     -0.001458
           JNJ    -0.000232
           PFE    -0.000366

## 4. XpyQ-EWMA inference (no saved model — always live)

In [5]:
xpyq_pred = pd.Series(np.nan, index=X_infer.index, name='xpyq_pred')

if RUN_XPYQ:
    XPYQ_KEY = os.environ.get('XPYQ_KEY') or os.environ.get('XPYQ_API_KEY')
    if not XPYQ_KEY:
        print('XPYQ_KEY not set; skipping')
    else:
        XPYQ_BASE = 'https://xpyq-lib-production.up.railway.app'
        H = {'Authorization': f'Bearer {XPYQ_KEY}', 'Content-Type': 'application/json'}

        returns_full = pd.read_parquet(os.path.join(DATA_DIR, 'returns.parquet'))
        # 60-day lookback before inference start
        lookback = returns_full.index[returns_full.index < target_dates[0]][-60:]
        all_inf_dates = list(lookback) + list(target_dates)
        returns_window = returns_full.loc[all_inf_dates].fillna(0).astype(np.float32)
        n_lookback = len(lookback)

        print(f'[xpyq] sending {returns_window.shape} returns matrix '
              f'({n_lookback} lookback + {len(target_dates)} inference dates)')

        code = f'''import numpy as _np
import json as _json
returns_arr = _np.array({returns_window.values.tolist()}, dtype=_np.float32)
R = from_numpy(returns_arr)
T, N = returns_arr.shape
cov = quant.covariance(R).numpy()
eigvals, eigvecs = linalg.eigh(from_numpy(cov))
eigvals = eigvals.numpy()
eigvecs = eigvecs.numpy()
factor = eigvecs[:, -1]
if factor.sum() < 0:
    factor = -factor
alpha = 0.10
ewma = _np.zeros_like(returns_arr)
ewma[0] = returns_arr[0]
for i in range(1, T):
    ewma[i] = alpha * returns_arr[i] + (1 - alpha) * ewma[i-1]
factor_signal = ewma @ factor
factor_signal_centered = factor_signal - _np.mean(factor_signal)
forecast = 5.0 * ewma + 0.3 * (factor_signal_centered[:, None] * factor[None, :])
forecast_test = forecast[{n_lookback}:]
print("DATA_START")
print(_json.dumps(forecast_test.tolist()))
print("DATA_END")
'''
        submit = requests.post(f'{XPYQ_BASE}/api/v1/compute/runs', headers=H,
                              json={'code': code, 'name': 'xpyq_infer'}, timeout=15)
        run_id = submit.json()['run_id']
        print(f'[xpyq] submitted run_id={run_id}')
        deadline = time.time() + 60
        poll = None
        while time.time() < deadline:
            try:
                resp = requests.get(f'{XPYQ_BASE}/api/v1/compute/runs/{run_id}', headers=H, timeout=10)
                if resp.status_code not in (200, 202):
                    time.sleep(1); continue
                p = resp.json()
                if not isinstance(p, dict):
                    time.sleep(1); continue
                poll = p
                if poll.get('status') in ('completed', 'failed', 'timed_out', 'cancelled'):
                    break
            except Exception:
                time.sleep(1); continue
            time.sleep(0.5)
        if isinstance(poll, dict) and poll.get('status') == 'completed':
            stdout = poll['stdout']
            payload = stdout.split('DATA_START')[1].split('DATA_END')[0].strip()
            forecast_arr = np.array(json.loads(payload), dtype=np.float32)
            forecast_df = pd.DataFrame(forecast_arr, index=target_dates, columns=returns_window.columns.tolist())
            forecast_long = forecast_df.stack()
            forecast_long.index = forecast_long.index.set_names(['timestamp', 'ticker'])
            xpyq_pred[:] = forecast_long.reindex(X_infer.index).values
            print(f'[xpyq] done. duration_ms={poll["duration_ms"]}, credits={poll["credits_charged"]}')
            display(xpyq_pred.head(10).to_frame())
        else:
            print(f'[xpyq] {poll["status"]}: {(poll.get("stdout") or "")[:300]}')
else:
    print('XpyQ disabled')

[xpyq] sending (90, 20) returns matrix (60 lookback + 30 inference dates)
[xpyq] submitted run_id=33dd050b-4529-4666-8a83-37d0c82f1e5f
[xpyq] failed: 


## 5. Combine predictions + emit signal

In [6]:
alpha_predictions = pd.DataFrame({
    'lightgbm':  lgb_pred,
    'vqc':       vqc_pred,
    'xpyq_ewma': xpyq_pred,
})

# Ensemble = mean of available model predictions
alpha_predictions['ensemble'] = alpha_predictions.mean(axis=1, skipna=True)

print(f'Inference predictions: {alpha_predictions.shape}')
display(alpha_predictions.head(15))

# Save
out_path = os.path.join(DATA_DIR, 'alpha_inference.parquet')
alpha_predictions.to_parquet(out_path)
alpha_predictions.to_csv(out_path.replace('.parquet', '.csv'))
print(f'\nSaved -> {out_path}')
print(f'Saved -> {out_path.replace(".parquet", ".csv")}')

# Per-day ranking: who to long, who to short
print('\n=== Sample: ensemble alpha ranking for the most recent inference date ===')
last_date = alpha_predictions.index.get_level_values('timestamp').max()
today_view = alpha_predictions.xs(last_date, level='timestamp').sort_values('ensemble', ascending=False)
print(f'\nDate: {last_date.date()}')
print('LONG (top 5 by ensemble):')
display(today_view.head(5))
print('SHORT (bottom 5 by ensemble):')
display(today_view.tail(5))

Inference predictions: (600, 4)


lightgbm       vqc  xpyq_ewma  ensemble
timestamp  ticker                                         
2020-04-29 AAPL    0.000926 -0.001095        NaN -0.000085
           MSFT   -0.007057 -0.000908        NaN -0.003982
           GOOGL   0.009064 -0.001061        NaN  0.004001
           NVDA    0.064054  0.001706        NaN  0.032880
           JPM     0.016931  0.000382        NaN  0.008656
           BAC     0.015187  0.000983        NaN  0.008085
           V       0.030086 -0.001460        NaN  0.014313
           MA      0.015338 -0.001458        NaN  0.006940
           JNJ     0.013045 -0.000232        NaN  0.006407
           PFE     0.002397 -0.000366        NaN  0.001015
           UNH     0.023850 -0.001653        NaN  0.011098
           LLY     0.024740  0.000073        NaN  0.012406
           WMT     0.009714  0.000314        NaN  0.005014
           PG      0.009485  0.000049        NaN  0.004767
           KO      0.010003  0.000051        NaN  0.005027


Saved -> /Applications/Projects/Hackathons/quantum_ai/workspace/data/alpha_inference.parquet
Saved -> /Applications/Projects/Hackathons/quantum_ai/workspace/data/alpha_inference.csv

=== Sample: ensemble alpha ranking for the most recent inference date ===

Date: 2020-06-10
LONG (top 5 by ensemble):


,lightgbm,vqc,xpyq_ewma,ensemble
ticker,,,,
BA,0.022454,0.007546,NaN,0.015000
XOM,0.025651,0.001539,NaN,0.013595
CVX,0.025692,-0.001741,NaN,0.011975
NVDA,0.019741,0.002674,NaN,0.011208
BAC,0.018973,-0.001436,NaN,0.008769


SHORT (bottom 5 by ensemble):


,lightgbm,vqc,xpyq_ewma,ensemble
ticker,,,,
PG,-0.005493,-0.000161,NaN,-0.002827
JNJ,-0.003463,-0.002306,NaN,-0.002884
WMT,-0.003531,-0.002791,NaN,-0.003161
LLY,-0.008025,0.000276,NaN,-0.003875
MSFT,-0.008142,-0.001124,NaN,-0.004633
